## Academic portfolio context

**Student / portfolio maintainer:** Mohammed Alaa Eddine GUERBOUZ  
**Institution:** INGETIS (Paris, France)  
**Programme:** Master-level studies · Generative AI / Large Language Models  
**Academic year:** 2026–2027  
**Topic:** Text Classification with Embeddings and Keras  
**Material type:** Course practical notebook / learning exercise  

**Problem explored:** Can pretrained text embeddings support a lightweight supervised classifier without training an LLM?

> Portfolio annotation added in October 2026. Original instructional content and attribution are preserved below. This header identifies the student maintaining this study portfolio; it is **not** a claim that all classroom template code was originally authored by the student. The source filename was `jour_2_classification_embeddings_keras_colab_fr.ipynb`. See the folder README for requirements and limitations.


# Jour 2 - Classifier des embeddings avec Keras et l'API Gemini

## Présentation

Bienvenue dans ce nouveau notebook du cours sur l'IA générative. Vous allez apprendre à utiliser les embeddings produits par l'API Gemini pour entraîner un modèle capable de classer des messages de forums (newsgroups) dans leur catégorie (le forum d'origine), à partir de leur contenu.

Cette technique utilise les embeddings de l'API Gemini comme entrée, ce qui évite d'entraîner un modèle directement sur du texte. Résultat : elle donne de bons résultats avec relativement peu d'exemples, comparé à l'entraînement d'un modèle de texte à partir de zéro.

> **Version adaptée pour Google Colab** : ce notebook est en français, lit la clé API depuis les **secrets Colab**, relance automatiquement les appels en cas de quota atteint et utilise le modèle `gemini-embedding-001` (le modèle `text-embedding-004` d'origine a été arrêté le 14 janvier 2026). Les embeddings sont calculés **par lots** pour limiter le nombre de requêtes.

## Installer le SDK

In [ ]:
# Colab inclut déjà google-genai : on le met simplement à jour vers la dernière version
# (nécessaire pour les modèles Gemini récents).
!pip install -U -q "google-genai"

In [ ]:
from google import genai
from google.genai import types

genai.__version__

# Modèle d'embedding utilisé dans tout le notebook.
EMBEDDING_MODEL = "gemini-embedding-001"

### Configurer votre clé API

Pour exécuter la cellule suivante, votre clé API doit être enregistrée dans un [secret Colab](https://colab.research.google.com/notebooks/secrets.ipynb) nommé `GOOGLE_API_KEY` :

1. Dans la barre latérale gauche de Colab, cliquez sur l'icône **clé** 🔑 (« Secrets »).
2. Cliquez sur **Ajouter un secret**, saisissez `GOOGLE_API_KEY` comme nom et collez votre clé comme valeur.
3. Activez l'interrupteur **Accès au notebook**.

Si vous n'avez pas encore de clé API, vous pouvez en obtenir une sur [AI Studio](https://aistudio.google.com/app/apikey). Vous trouverez des [instructions détaillées dans la documentation](https://ai.google.dev/gemini-api/docs/api-key).

Hors de Colab (exécution locale, VM…), la cellule utilise la variable d'environnement `GOOGLE_API_KEY`.

In [ ]:
import os

GOOGLE_API_KEY = None

# 1) Secrets Colab (icône clé dans la barre latérale gauche)
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # On n'est pas dans Colab

# 2) Variable d'environnement (exécution locale, VM, etc.)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "Clé API introuvable : ajoutez un secret Colab nommé GOOGLE_API_KEY "
        "et activez « Accès au notebook »."
    )

client = genai.Client(api_key=GOOGLE_API_KEY)
print("Clé API chargée ✅")

Si vous obtenez une erreur du type `SecretNotFoundError` ou `NotebookAccessError`, vérifiez que le secret s'appelle exactement `GOOGLE_API_KEY` **et** que l'interrupteur **Accès au notebook** est activé dans le panneau Secrets (icône clé). Colab peut aussi afficher une fenêtre vous demandant d'autoriser l'accès : cliquez sur **Autoriser l'accès**.

## Jeu de données

Le jeu de données [20 Newsgroups](https://scikit-learn.org/stable/datasets/real_world.html#newsgroups-dataset) contient 18 000 messages de forums sur 20 sujets, répartis en un ensemble d'entraînement et un ensemble de test. La séparation entre les deux se fait selon la date : messages publiés avant et après une date donnée. Dans ce tutoriel, vous utiliserez des sous-échantillons de ces deux ensembles, avec un peu de traitement dans Pandas.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(subset="train")
newsgroups_test = fetch_20newsgroups(subset="test")

# Liste des noms de classes du jeu de données
newsgroups_train.target_names

Voici à quoi ressemble un enregistrement de l'ensemble d'entraînement.

In [ ]:
print(newsgroups_train.data[0])

Commencez par prétraiter les données dans un dataframe Pandas. Pour retirer les informations sensibles comme les noms et les adresses e-mail, vous ne garderez que le sujet et le corps de chaque message. Cette étape facultative transforme les données en texte plus générique (plutôt que des e-mails), pour qu'elles soient utilisables dans d'autres contextes.

In [ ]:
import email
import re

import pandas as pd


def preprocess_newsgroup_row(data):
    # Garde uniquement le sujet et le corps du message
    msg = email.message_from_string(data)
    text = f"{msg['Subject']}\n\n{msg.get_payload()}"
    # Supprime les adresses e-mail restantes
    text = re.sub(r"[\w\.-]+@[\w\.-]+", "", text)
    # Tronque chaque entrée à 5 000 caractères
    text = text[:5000]

    return text


def preprocess_newsgroup_data(newsgroup_dataset):
    # Place les données dans un dataframe
    df = pd.DataFrame(
        {"Text": newsgroup_dataset.data, "Label": newsgroup_dataset.target}
    )
    # Nettoie le texte
    df["Text"] = df["Text"].apply(preprocess_newsgroup_row)
    # Associe chaque label au nom de sa classe
    df["Class Name"] = df["Label"].map(lambda l: newsgroup_dataset.target_names[l])

    return df

In [ ]:
# Applique le prétraitement aux ensembles d'entraînement et de test
df_train = preprocess_newsgroup_data(newsgroups_train)
df_test = preprocess_newsgroup_data(newsgroups_test)

df_train.head()

Ensuite, vous allez échantillonner les données : un nombre fixe d'exemples par catégorie dans l'ensemble d'entraînement, en ne gardant que quelques catégories pour ce tutoriel. Choisissez les catégories scientifiques (`sci`) à comparer.

In [ ]:
def sample_data(df, num_samples, classes_to_keep):
    # Échantillonne num_samples lignes pour chaque label.
    df = (
        df.groupby("Label")[df.columns]
        .apply(lambda x: x.sample(num_samples))
        .reset_index(drop=True)
    )

    df = df[df["Class Name"].str.contains(classes_to_keep)]

    # Il y a moins de catégories : on recalcule l'encodage des labels.
    df["Class Name"] = df["Class Name"].astype("category")
    df["Encoded Label"] = df["Class Name"].cat.codes

    return df

In [ ]:
# Valeurs d'origine : 100 exemples d'entraînement et 25 de test par catégorie.
# Elles sont réduites ici pour ménager le quota des clés gratuites (tokens par minute).
# Avec une clé payante, vous pouvez revenir à 100 et 25.
TRAIN_NUM_SAMPLES = 50
TEST_NUM_SAMPLES = 10
# Le nom de classe doit contenir « sci » pour garder les catégories scientifiques.
# Essayez d'autres labels : voir newsgroups_train.target_names
CLASSES_TO_KEEP = "sci"

df_train = sample_data(df_train, TRAIN_NUM_SAMPLES, CLASSES_TO_KEEP)
df_test = sample_data(df_test, TEST_NUM_SAMPLES, CLASSES_TO_KEEP)

In [ ]:
df_train.value_counts("Class Name")

In [ ]:
df_test.value_counts("Class Name")

## Créer les embeddings

Dans cette section, vous allez générer un embedding pour chaque texte avec le point d'accès d'embeddings de l'API Gemini. Pour en savoir plus, consultez le [guide des embeddings](https://ai.google.dev/gemini-api/docs/embeddings).

**Remarque** : dans le notebook d'origine, les embeddings étaient calculés un par un. Ici, ils sont envoyés **par lots de 50 textes** : on fait donc beaucoup moins de requêtes, mais le nombre de tokens reste le même. Avec une clé gratuite, cette étape peut prendre quelques minutes si le quota de tokens par minute est atteint : le retry attend alors automatiquement.

### Types de tâche

Le modèle `gemini-embedding-001` accepte un paramètre de type de tâche (`task_type`), qui produit des embeddings adaptés à la tâche visée.

Type de tâche | Description
---       | ---
RETRIEVAL_QUERY	| Le texte est une requête dans un contexte de recherche/récupération.
RETRIEVAL_DOCUMENT | Le texte est un document dans un contexte de recherche/récupération.
SEMANTIC_SIMILARITY	| Le texte servira à mesurer la similarité sémantique (STS).
CLASSIFICATION	| Les embeddings serviront à de la classification.
CLUSTERING	| Les embeddings serviront à du regroupement (clustering).
FACT_VERIFICATION | Le texte servira à vérifier des faits.

Dans cet exemple, vous allez faire de la classification.

La politique de retry est un peu plus patiente que dans les autres notebooks (jusqu'à 5 minutes), car le quota de tokens par minute se réinitialise au bout d'une minute.

In [ ]:
from google.api_core import retry
import tqdm
from tqdm.rich import tqdm as tqdmr
import warnings

# Ajoute tqdm à Pandas...
tqdmr.pandas()

# ...mais masque l'avertissement « expérimental ».
warnings.filterwarnings("ignore", category=tqdm.TqdmExperimentalWarning)

# Relance automatique en cas de quota atteint (429) ou de modèle surchargé (503) :
# 2 s, 4 s, 8 s... (30 s max entre deux essais), 5 minutes au total.
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})
politique_retry = retry.Retry(predicate=is_retriable, initial=2, maximum=30, timeout=300)

TAILLE_LOT = 50  # Nombre de textes envoyés par requête


@politique_retry
def embed_batch(texts: list[str]) -> list[list[float]]:
    """Calcule les embeddings d'une liste de textes en une seule requête."""
    # On fait de la classification : on règle task_type en conséquence.
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type="classification",
        ),
    )
    return [e.values for e in response.embeddings]


def embed_fn(text: str) -> list[float]:
    """Calcule l'embedding d'un seul texte."""
    return embed_batch([text])[0]


def create_embeddings(df):
    textes = df["Text"].tolist()
    embeddings = []
    for debut in tqdm.tqdm(range(0, len(textes), TAILLE_LOT), desc="Lots"):
        embeddings.extend(embed_batch(textes[debut:debut + TAILLE_LOT]))
    df["Embeddings"] = embeddings
    return df

Ce code privilégie la lisibilité. Pour aller plus loin, vous pouvez essayer une génération d'embeddings parallèle ou asynchrone (`client.aio`), ou l'[API Batch](https://ai.google.dev/gemini-api/docs/batch-api), moins chère mais asynchrone. Cette étape prend un peu de temps.

In [ ]:
df_train = create_embeddings(df_train)
df_test = create_embeddings(df_test)

In [ ]:
df_train.head()

## Construire un modèle de classification

Vous allez définir un modèle simple qui prend en entrée les embeddings bruts, avec une couche cachée et une couche de sortie qui donne les probabilités de chaque classe. La prédiction correspond à la probabilité qu'un texte appartienne à une catégorie donnée.

Pendant l'exécution, Keras s'occupe des détails comme le mélange des données, le calcul des métriques et le reste du code habituel de ML.

In [ ]:
import keras
from keras import layers


def build_classification_model(input_size: int, num_classes: int) -> keras.Model:
    return keras.Sequential(
        [
            layers.Input([input_size], name="embedding_inputs"),
            layers.Dense(input_size, activation="relu", name="hidden"),
            layers.Dense(num_classes, activation="softmax", name="output_probs"),
        ]
    )

In [ ]:
# La taille de l'embedding est déduite des données (3 072 pour gemini-embedding-001).
# Vous pouvez aussi la réduire avec le paramètre `output_dimensionality` de `embed_content`.
embedding_size = len(df_train["Embeddings"].iloc[0])

classifier = build_classification_model(
    embedding_size, len(df_train["Class Name"].unique())
)
classifier.summary()

classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

## Entraîner le modèle

Vous pouvez enfin entraîner votre modèle. Ce code utilise l'arrêt anticipé (*early stopping*) pour sortir de la boucle d'entraînement dès que la précision se stabilise : le nombre d'époques réellement exécutées peut donc être inférieur à la valeur indiquée.

In [ ]:
import numpy as np


NUM_EPOCHS = 20
BATCH_SIZE = 32

# Sépare les composantes x et y des ensembles d'entraînement et de validation.
y_train = df_train["Encoded Label"]
x_train = np.stack(df_train["Embeddings"])
y_val = df_test["Encoded Label"]
x_val = np.stack(df_test["Embeddings"])

# Autorise un arrêt anticipé si la précision se stabilise.
early_stop = keras.callbacks.EarlyStopping(monitor="accuracy", patience=3)

# Entraîne le modèle sur le nombre d'époques voulu.
history = classifier.fit(
    x=x_train,
    y=y_train,
    validation_data=(x_val, y_val),
    callbacks=[early_stop],
    batch_size=BATCH_SIZE,
    epochs=NUM_EPOCHS,
)

## Évaluer les performances du modèle

Utilisez <a href="https://keras.io/api/models/model_training_apis/#evaluate-method"><code>Model.evaluate</code></a> de Keras pour calculer la perte et la précision sur l'ensemble de test.

In [ ]:
classifier.evaluate(x=x_val, y=y_val, return_dict=True)

Pour en savoir plus sur l'entraînement de modèles avec Keras, notamment la visualisation des métriques d'entraînement, lisez [Training & evaluation with built-in methods](https://keras.io/guides/training_with_built_in_methods/).

## Essayer une prédiction personnalisée

Maintenant que vous avez un modèle entraîné avec de bonnes métriques, essayez une prédiction sur de nouvelles données écrites à la main. Utilisez l'exemple fourni, ou vos propres données, pour voir comment le modèle se comporte.

In [ ]:
def make_prediction(text: str) -> list[float]:
    """Prédit les catégories du texte fourni."""
    # Le modèle prend des embeddings en entrée : on les calcule d'abord.
    embedded = embed_fn(text)

    # L'entrée doit être un lot (batch) : on l'enveloppe dans une liste
    # pour obtenir un lot de taille 1.
    inp = np.array([embedded])

    # Et on retire la dimension du lot ici.
    [result] = classifier.predict(inp)
    return result

L'exemple ci-dessous est en anglais, comme les données d'entraînement. En substance : « Débutant qui cherche à partir d'ici. Je voudrais voyager jusqu'aux confins ! Quel engin puis-je acheter ? Quel est le plus facile d'accès depuis ce 3e rocher ? »

In [ ]:
# Cet exemple évite tout vocabulaire spécifique à l'espace, pour voir si le modèle
# ne se contente pas de repérer du jargon.
new_text = """
First-timer looking to get out of here.

Hi, I'm writing about my interest in travelling to the outer limits!

What kind of craft can I buy? What is easiest to access from this 3rd rock?

Let me know how to do that please.
"""

result = make_prediction(new_text)

for idx, category in enumerate(df_test["Class Name"].cat.categories):
    print(f"{category}: {result[idx] * 100:0.2f}%")